# Public source archive

Recovered historical code, prepared for public review. AI assisted the original coding and this archive preparation; team projects retain team attribution in the project README.

Outputs, notebook attachments, personal metadata, and embedded organizer datasets have been removed. Narrative cells retain headings only. Supply authorized input files before running data-dependent notebooks. This archive does not claim that manual checks were completed or that the code was independently developed.


In [ ]:
import json, os
EXCLUDED_JOB_IDS = json.loads(os.environ.get("EXCLUDED_JOB_IDS", "[]"))
from pathlib import Path
import re, json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
from html import escape
DATA = Path('/content')
OUT = DATA / 'hackathon_results'
OUT.mkdir(exist_ok=True)
COMPANIES = ['Accenture', 'Booz Allen Hamilton', 'Capgemini']
FOCAL = 'Accenture'
assert FOCAL in COMPANIES
NAMES = ['organizations.csv', 'job_postings.csv', 'reviews.csv', 'review_job_role_links.csv']

def candidates(name):
    return [p for p in DATA.glob('*.csv') if p.name == name or p.name.endswith('-' + name)]
if any((not candidates(name) for name in NAMES)):
    from google.colab import files
    print('请一次上传老师的四个原始CSV文件。')
    files.upload()
paths = {}
for name in NAMES:
    found = candidates(name)
    if len(found) != 1:
        raise ValueError(f'{name}找到{len(found)}份。请保留一份原始文件后重跑。')
    paths[name] = found[0]
raw_tables = {name: pd.read_csv(p, dtype=str, encoding='utf-8-sig').fillna('') for name, p in paths.items()}
org = raw_tables['organizations.csv']
selected_org = org[org['organization_name'].isin(COMPANIES)]
assert set(selected_org['organization_name']) == set(COMPANIES)
org_ids = set(selected_org['organization_id'])

def select_unique(name, id_col):
    frame = raw_tables[name]
    frame = frame[frame['organization_id'].isin(org_ids)].copy()
    frame = frame.drop_duplicates()
    if frame[id_col].eq('').any() or frame[id_col].duplicated().any():
        raise ValueError(f'{name}存在缺失或冲突ID，需要先检查。')
    return frame
J = select_unique('job_postings.csv', 'job_id')
R = select_unique('reviews.csv', 'review_id')
L = raw_tables['review_job_role_links.csv']
L = L[L['organization_id'].isin(org_ids)].drop_duplicates().copy()
assert L['job_id'].isin(J['job_id']).all()
assert L['review_id'].isin(R['review_id']).all()
J['attribution_pending'] = J['job_id'].isin(EXCLUDED_JOB_IDS)
J_main = J[~J['attribution_pending']].copy()

def scope_table():
    rows = []
    for company in COMPANIES:
        a = J[J.organization_name.eq(company)]
        b = R[R.organization_name.eq(company)]
        rows.append({'公司': company, '原始广告数': len(a), '主比较广告数': int((~a.attribution_pending).sum()), '评论数': len(b), '广告不同标题数': a.title.nunique(), '评论岗位缺失数': int(b.reviewer_job_title.eq('').sum()), '广告重复正文行数': int(a.loc[a.text_sha256.ne(''), 'text_sha256'].duplicated().sum()), '评论重复正文行数': int(b.loc[b.text_sha256.ne(''), 'text_sha256'].duplicated().sum())})
    return pd.DataFrame(rows).set_index('公司')
scope = scope_table()
display(scope)
manifest = {'question': 'Improve employment offering and recruitment messaging for early-career talent', 'focal_employer': FOCAL, 'companies': COMPANIES, 'input_sha256': {n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in paths.items()}, 'python': platform.python_version(), 'pandas': pd.__version__, 'rules_version': 'v1_frozen_before_manual_check', 'exclusion': {}, 'notes': 'Historical snapshots; linked rows are comparable roles, not confirmed same openings.'}
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
THEMES = {
    'Career development': r'\btrain[A-Za-z_]*\b|\blearn[A-Za-z_]*\b|\bmentor[A-Za-z_]*\b|\bcoach[A-Za-z_]*\b|\bonboard[A-Za-z_]*\b|\bupskill[A-Za-z_]*\b|\bcareer\b|\bpromot[A-Za-z_]*\b|\bprogression\b|\btuition\b|\bcourses?\b',
    'Flexibility & workload': r'\bhybrid\b|\bremote\b|\bflexib[A-Za-z_]*\b|\bwork.from.home\b|\bwfh\b|\bwork.life\b|\bworkload\b|\bovertime\b|\blong hours\b|\brotational shifts?\b|\bburnout\b|\bburn.out\b',
    'Compensation & benefits': r'\bsalar[A-Za-z_]*\b|\bpay\b|\bcompensation\b|\bbenefits?\b|\bbonus[A-Za-z_]*\b|\binsurance\b|\bretirement\b|\b401\(?k\)?\b|\bpension\b|\bpaid leave\b|\bpaid time off\b|\bhikes?\b'
}
TITLE_SIGNAL = r'\bintern(?:s|ship|ships)?\b|\bjunior\b|\bgraduates?\b|\bentry[ -]level\b|\btrainees?\b|\bapprentices?\b'
BODY_SIGNAL = TITLE_SIGNAL + r'|\bfreshers?\b|\bearly[ -]career\b|\bfirst job\b|\bnew joiners?\b'

def plain(s):
    return ' '.join(str(s).split())

def theme_flag(s, pattern):
    cleaned = re.sub(r'\b(?:machine|deep)[ -]learning\b', '', plain(s), flags=re.I)
    return bool(re.search(pattern, cleaned, flags=re.I))

for frame in [J, J_main]:
    frame['text'] = frame.description
    frame['early_title_signal'] = frame.title.str.contains(TITLE_SIGNAL, case=False, regex=True)
    frame['early_body_signal'] = frame.description.str.contains(BODY_SIGNAL, case=False, regex=True)
    frame['early_signal'] = frame[['early_title_signal', 'early_body_signal']].any(axis=1)
    for theme, pattern in THEMES.items():
        frame[theme] = frame.text.map(lambda value, p=pattern: theme_flag(value, p))

def theme_summary(frame):
    rows = []
    for company in COMPANIES:
        group = frame[frame.organization_name.eq(company)]
        for theme in THEMES:
            n, count = len(group), int(group[theme].sum())
            rows.append({'company':company, 'theme':theme, 'n':n,
                         'mentions':count, 'percent':100*count/n if n else np.nan})
    return pd.DataFrame(rows)

ad_baseline = theme_summary(J_main)
def formatted_summary(table):
    temp = table.copy()
    temp['count/n (%)'] = temp.apply(
        lambda x: f"{int(x['mentions'])}/{int(x['n'])} ({x['percent']:.1f}%)" if x['n'] else 'N/A', axis=1)
    return temp.pivot(index='company', columns='theme', values='count/n (%)').reindex(COMPANIES)

display(formatted_summary(ad_baseline))
ad_baseline.to_csv(OUT/'postings_only_baseline.csv', index=False)
print('仅为文本提及基线。初级标题或正文线索仍须读资格条件确认；其他广告作为组织背景。')

In [ ]:
FIELDS = ['summary', 'pros', 'cons', 'advice_to_management']
R['text'] = R[FIELDS].agg('\n'.join, axis=1)
R['early_title_signal'] = R.reviewer_job_title.str.contains(TITLE_SIGNAL, case=False, regex=True)
R['early_body_signal'] = R.text.str.contains(BODY_SIGNAL, case=False, regex=True)
R['early_signal'] = R[['early_title_signal', 'early_body_signal']].any(axis=1)
for theme, pattern in THEMES.items():
    R[theme] = R.text.map(lambda value, p=pattern: theme_flag(value, p))
review_summary = theme_summary(R)
early_review_summary = theme_summary(R[R.early_signal])
display(formatted_summary(review_summary))

evidence_rows = []
for source, frame, identifier, title_col, fields in [
    ('posting', J, 'job_id', 'title', ['description']),
    ('review', R, 'review_id', 'reviewer_job_title', FIELDS)
]:
    for _, row in frame.iterrows():
        for field in fields:
            value = plain(row[field])
            for theme, pattern in THEMES.items():
                if not theme_flag(value, pattern):
                    continue
                evidence_rows.append({
                    'source':source, 'record_id':row[identifier],
                    'company':row.organization_name, 'title':row[title_col],
                    'early_signal':row.early_signal, 'theme':theme,
                    'field':field, 'full_field_text':value,
                    'attribution_pending':bool(row.get('attribution_pending', False)),
                    'human_status':'pending'
                })
evidence = pd.DataFrame(evidence_rows)
evidence.to_csv(OUT/'evidence_index.csv', index=False)

def show_evidence(company=FOCAL, source='review', theme='Career development', early_only=False, limit=5):
    subset = evidence[(evidence.company.eq(company)) & (evidence.source.eq(source)) & (evidence.theme.eq(theme))]
    if early_only:
        subset = subset[subset.early_signal]
    print(f'符合条件的字段片段：{len(subset)}；独立记录：{subset.record_id.nunique()}。只显示前{limit}条。')
    html = []
    for _, row in subset.sort_values(['record_id','field']).head(limit).iterrows():
        heading = f'{row.company} | {row.source} {row.record_id} | {row.title} | {row.field}'
        html.append(f'<details><summary>{escape(heading)}</summary><p>{escape(row.full_field_text)}</p></details>')
    display(HTML('<br>'.join(html)))

show_evidence()
# 例：show_evidence('Capgemini', 'posting', 'Flexibility & workload', False, 5)
# 三个theme名称见THEMES；early_only=True仅表示带初级线索的候选记录。

4. 四张可用于汇报的基础图
作用：分别展示样本量、广告主题、评论主题，以及职业早期线索子集。图2与图3不可相减为“承诺落差”；两种来源、岗位和时间并未匹配。

图中使用英文避免Colab中文字体缺失。图2、3、4均须注明是词典提及率，待人工核查后决定是否用于最终PPT。

In [ ]:
plt.rcParams.update({'figure.dpi':120, 'font.size':10})
COLORS = ['#3659A2', '#26A69A', '#F4A340']

# 图1：样本量，广告和评论分别计数
counts = pd.DataFrame({
    'Postings (main comparison)': J_main.groupby('organization_name').size(),
    'Reviews': R.groupby('organization_name').size()
}).reindex(COMPANIES)
ax = counts.plot.bar(figsize=(9,4), color=COLORS[:2], rot=0)
ax.set(title='1. Source-specific sample sizes', xlabel='', ylabel='Distinct records')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'01_sample_sizes.png'); plt.show()

def theme_plot(table, title, filename):
    wide = table.pivot(index='company', columns='theme', values='percent').reindex(index=COMPANIES, columns=list(THEMES))
    ax = wide.plot.bar(figsize=(10,4), color=COLORS, rot=0)
    ax.set(title=title, xlabel='', ylabel='Records mentioning theme (%)', ylim=(0,105))
    ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5,-0.13), ncol=3)
    for i, company in enumerate(COMPANIES):
        n = int(table.loc[table.company.eq(company), 'n'].iloc[0])
        ax.text(i, 102, f'n={n}', ha='center', fontsize=8)
    plt.tight_layout(); plt.savefig(OUT/filename, bbox_inches='tight'); plt.show()

theme_plot(ad_baseline, '2. Posting theme mentions — not verified promises', '02_posting_mentions.png')
theme_plot(review_summary, '3. Review theme mentions — not sentiment', '03_review_mentions.png')
theme_plot(early_review_summary, '4. Reviews with early-career signals — eligibility unverified', '04_early_signal_reviews.png')

In [ ]:
import ipywidgets as widgets

validation_records = []
for source, frame, identifier, title_col in [
    ('posting',J_main,'job_id','title'), ('review',R,'review_id','reviewer_job_title')
]:
    for company in COMPANIES:
        group = frame[frame.organization_name.eq(company)].sort_values(identifier)
        parts = []
        for flag, count in [(True,3), (False,2)]:
            pool = group[group.early_signal.eq(flag)]
            parts.append(pool.sample(min(count,len(pool)), random_state=2026))
        sample = pd.concat(parts)
        if len(sample) < min(5,len(group)):
            rest = group[~group[identifier].isin(sample[identifier])]
            sample = pd.concat([sample,rest.sample(min(5-len(sample),len(rest)),random_state=2026)])
        for _, row in sample.iterrows():
            record = {'source':source, 'record_id':row[identifier],
                      'company':company, 'title':row[title_col],
                      'text':row.text, 'early_signal':bool(row.early_signal)}
            record['text_hash'] = hashlib.sha256(row.text.encode()).hexdigest()
            for theme in THEMES:
                record['pred_' + theme] = bool(row[theme])
            validation_records.append(record)

validation = pd.DataFrame(validation_records)
validation['key'] = validation.source + ':' + validation.record_id
check_path = OUT/'manual_checks.csv'
if 'manual_checks' not in globals():
    manual_checks = pd.read_csv(check_path,dtype=str).fillna('') if check_path.exists() else pd.DataFrame()

choice = widgets.Dropdown(options=[(f'{i+1}. {row.company} | {row.source} {row.record_id}',i)
                                  for i,row in validation.iterrows()], description='记录')
labels = {theme:widgets.Dropdown(options=['待核查','提及','未提及','无法判断'],description=theme[:12],
                               layout=widgets.Layout(width='450px')) for theme in THEMES}
early_label = widgets.Dropdown(options=['待核查','相关','不相关','无法判断'],description='初级相关')
kind = widgets.Dropdown(options=['待核查','具体条件或支持安排','通用资源或宣传','职责或技能','员工体验','混合','未涉及','无法判断'],description='证据性质')
notes = widgets.Textarea(description='解释及依据',layout=widgets.Layout(width='95%',height='100px'))
reviewer = widgets.Text(description='核查人')
save_button = widgets.Button(description='保存并下一条',button_style='success')
eval_button = widgets.Button(description='查看核查进度及词典表现')
download_button = widgets.Button(description='下载核查CSV')
reader, messages = widgets.Output(), widgets.Output()

def load_record(change=None):
    row = validation.iloc[choice.value]
    previous = pd.DataFrame()
    if not manual_checks.empty:
        previous = manual_checks[(manual_checks.key.eq(row.key)) & (manual_checks.text_hash.eq(row.text_hash))]
    old = previous.iloc[-1].to_dict() if len(previous) else {}
    for theme, control in labels.items():
        control.value = old.get('human_' + theme,'待核查')
    early_label.value = old.get('human_early','待核查')
    kind.value = old.get('evidence_kind','待核查')
    notes.value = old.get('notes','')
    reviewer.value = old.get('reviewer',reviewer.value)
    with reader:
        clear_output(wait=True)
        print(row.title)
        print('以下是原文。review各字段的完整分界见R表；程序不会自动推断情绪。')
        if row.source == 'review':
            original = R[R.review_id.eq(row.record_id)].iloc[0]
            for field in FIELDS:
                print('\n' + field + ':\n' + original[field])
        else:
            print(row.text)

def save_check(button):
    global manual_checks
    with messages:
        clear_output()
        if (any(c.value == '待核查' for c in labels.values()) or early_label.value == '待核查'
            or kind.value == '待核查' or not reviewer.value.strip() or not notes.value.strip()):
            print('请填写三个主题判断、初级相关性、证据性质、依据和核查人后保存。'); return
        row = validation.iloc[choice.value]
        record = {'key':row.key,'source':row.source,'record_id':row.record_id,
                  'company':row.company,'text_hash':row.text_hash,
                  'human_early':early_label.value,'evidence_kind':kind.value,
                  'notes':notes.value,'reviewer':reviewer.value}
        record.update({'human_'+t:c.value for t,c in labels.items()})
        if not manual_checks.empty:
            manual_checks = manual_checks[~manual_checks.key.eq(row.key)]
        manual_checks = pd.concat([manual_checks,pd.DataFrame([record])],ignore_index=True)
        manual_checks.to_csv(check_path,index=False)
        print(f'已保存；本会话文件中共有{len(manual_checks)}条记录。')
    if choice.value < len(validation)-1:
        choice.value += 1

def evaluate_checks(button=None):
    with messages:
        clear_output()
        if manual_checks.empty:
            print('人工核查完成0条。'); return
        checked = validation.merge(manual_checks,on=['key','text_hash'],suffixes=('','_manual'))
        print(f'当前规则及原文匹配的已核查记录：{len(checked)}/{len(validation)}。')
        rows = []
        for source in ['posting','review']:
            for theme in THEMES:
                g = checked[(checked.source.eq(source)) & checked['human_'+theme].isin(['提及','未提及'])]
                p, y = g['pred_'+theme].astype(bool), g['human_'+theme].eq('提及')
                tp, fp, fn = int((p & y).sum()), int((p & ~y).sum()), int((~p & y).sum())
                precision = tp/(tp+fp) if tp+fp else np.nan
                recall = tp/(tp+fn) if tp+fn else np.nan
                f1 = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan
                rows.append({'source':source,'theme':theme,'evaluated_n':len(g),
                             'TP':tp,'FP':fp,'FN':fn,'precision':precision,'recall':recall,'F1':f1})
        result = pd.DataFrame(rows)
        display(result.round(3))
        result.to_csv(OUT/'validation_diagnostics.csv',index=False)
        print('仅为分层核查样本的诊断；无法判断项不参与二分类指标。')

def download_checks(button):
    with messages:
        clear_output()
        if not check_path.exists():
            print('请先保存至少一条核查记录。'); return
        from google.colab import files
        files.download(str(check_path))

choice.observe(load_record,names='value')
save_button.on_click(save_check)
eval_button.on_click(evaluate_checks)
download_button.on_click(download_checks)
display(widgets.VBox([choice,reader,*labels.values(),early_label,kind,notes,reviewer,
                      widgets.HBox([save_button,eval_button,download_button]),messages]))
load_record()

7. 回到老师的问题：两项优先行动和三个学生面试问题
**不要按关键词频次自动选建议。**结合重点公司的具体广告承诺、正面及负面评论、同行案例、岗位地区时间差异，以及人工核查结果，选两项有业务意义且可以执行的措施。

每条发现标记为：
- Alignment（一致）：两种来源在主题上提供相容的信息，仍不代表同一岗位得到验证。
- Potential tension（潜在张力）：广告安排与部分员工报告存在值得调查的差异，不能直接称为虚假宣传。
- Insufficient evidence（证据不足）：缺少清晰、可比的信息；未提及不等于不存在。
每项建议必须同时考虑实际工作机会/条件/福利与招聘表达，并写清支持证据、同行参考、不确定性、衡量指标。指标属于未来测量方案，本数据不包含招聘转化或留存结果，不能声称已验证有效。

可参考的衡量方式：项目匹配等待时间、前90天导师接触率、新人支持满意度、招聘页面信息理解率、合格申请或offer接受率。依具体行动选取，不必全部使用。

In [ ]:
recommendations = pd.DataFrame([
    {'priority':1,'focal_employer':FOCAL,'action':'待根据核查结果填写',
     'employment_offering_change':'','recruitment_message_change':'',
     'posting_ids':'','review_ids':'','peer_evidence_ids':'',
     'interpretation_label':'','uncertainty':'','future_metric_or_experiment':''},
    {'priority':2,'focal_employer':FOCAL,'action':'待根据核查结果填写',
     'employment_offering_change':'','recruitment_message_change':'',
     'posting_ids':'','review_ids':'','peer_evidence_ids':'',
     'interpretation_label':'','uncertainty':'','future_metric_or_experiment':''}
])
display(recommendations)
interview_questions = [
    '新人前90天有哪些培训、导师和实际项目安排？能否给一个最近的例子？',
    '这个岗位实际的办公地点、工作时间和高峰期负担怎样？项目变化时如何沟通？',
    '薪酬福利和晋升标准如何确定？哪些条件因地区、项目或员工资格而不同？'
]
for i,q in enumerate(interview_questions,1):
    print(f'{i}. {q}')
print('上面是初稿问题，可根据最终两项建议修改。建议表尚未填写时，分析未完成。')

In [ ]:
evaluate_checks()
recommendations = pd.DataFrame(columns=['优先级', '重点公司', '建议', '实际安排', '招聘表达', '广告证据ID', '评论证据ID', '同行参考', '解释', '不确定性', '未来衡量'])
with pd.option_context('display.max_colwidth', 100):
    display(recommendations)
recommendations.to_csv(OUT / 'recommendations_draft.csv', index=False, encoding='utf-8-sig')
